In [48]:
import pandas as pd
from datetime import datetime



# Data Loading and DF creation

In [49]:
player_awards = pd.read_csv("basketballPlayoffs/awards_players.csv")
players = pd.read_csv("basketballPlayoffs/players.csv")
coaches = pd.read_csv("basketballPlayoffs/coaches.csv")
player_teams = pd.read_csv("basketballPlayoffs/players_teams.csv")
postseason_matchups = pd.read_csv("basketballPlayoffs/series_post.csv")
postseason_team_record = pd.read_csv("basketballPlayoffs/teams_post.csv")
teams = pd.read_csv("basketballPlayoffs/teams.csv")

dfs = ["player_awards","players","coaches","player_teams","postseason_matchups","postseason_team_record","teams"]


    
# Duplicate Values Drop



In [50]:
player_awards.drop_duplicates()
players.drop_duplicates()
coaches.drop_duplicates()
player_teams.drop_duplicates()
postseason_matchups.drop_duplicates()
postseason_team_record.drop_duplicates()
teams.drop_duplicates()

,year,lgID,tmID,franchID,confID,divID,rank,playoff,seeded,firstRound,...,GP,homeW,homeL,awayW,awayL,confW,confL,min,attend,arena
0,9,WNBA,ATL,ATL,EA,NaN,7,N,0,NaN,...,34,1,16,3,14,2,18,6825,141379,Philips Arena
1,10,WNBA,ATL,ATL,EA,NaN,2,Y,0,L,...,34,12,5,6,11,10,12,6950,120737,Philips Arena
2,1,WNBA,CHA,CHA,EA,NaN,8,N,0,NaN,...,32,5,11,3,13,5,16,6475,90963,Charlotte Coliseum
3,2,WNBA,CHA,CHA,EA,NaN,4,Y,0,W,...,32,11,5,7,9,15,6,6500,105525,Charlotte Coliseum
4,3,WNBA,CHA,CHA,EA,NaN,2,Y,0,L,...,32,11,5,7,9,12,9,6450,106670,Charlotte Coliseum
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,6,WNBA,WAS,WAS,EA,NaN,5,N,0,NaN,...,34,10,7,6,11,9,11,6900,171501,Verizon Center
138,7,WNBA,WAS,WAS,EA,NaN,4,Y,0,L,...,34,13,4,5,12,12,8,6850,133255,Verizon Center
139,8,WNBA,WAS,WAS,EA,NaN,5,N,0,NaN,...,34,8,9,8,9,8,12,6900,133255,Verizon Center
140,9,WNBA,WAS,WAS,EA,NaN,6,N,0,NaN,...,34,6,11,4,13,6,14,6825,154637,Verizon Center


# Drop Unique Tables

In [51]:
for name in dfs:

    df = globals()[name]
    for col in df.columns:
        if len(df[col].unique()) == 1:
            df.drop(col, inplace=True, axis=1)
    
    print(f"Processed {name}: Remaining columns: {df.columns}")


Processed player_awards: Remaining columns: Index(['playerID', 'award', 'year'], dtype='object')
Processed players: Remaining columns: Index(['bioID', 'pos', 'height', 'weight', 'college', 'collegeOther',
       'birthDate', 'deathDate'],
      dtype='object')
Processed coaches: Remaining columns: Index(['coachID', 'year', 'tmID', 'stint', 'won', 'lost', 'post_wins',
       'post_losses'],
      dtype='object')
Processed player_teams: Remaining columns: Index(['playerID', 'year', 'stint', 'tmID', 'GP', 'GS', 'minutes', 'points',
       'oRebounds', 'dRebounds', 'rebounds', 'assists', 'steals', 'blocks',
       'turnovers', 'PF', 'fgAttempted', 'fgMade', 'ftAttempted', 'ftMade',
       'threeAttempted', 'threeMade', 'dq', 'PostGP', 'PostGS', 'PostMinutes',
       'PostPoints', 'PostoRebounds', 'PostdRebounds', 'PostRebounds',
       'PostAssists', 'PostSteals', 'PostBlocks', 'PostTurnovers', 'PostPF',
       'PostfgAttempted', 'PostfgMade', 'PostftAttempted', 'PostftMade',
       'Postt

# Player Column Cleaning Up


In [52]:
# Removing DeathDate to a binary isAlive
players['isAlive'] = players['deathDate'].apply(lambda x: 1 if x == "0000-00-00" else 0)
players.drop('deathDate', axis=1, inplace=True)

players['birthDate'] = pd.to_datetime(players['birthDate'], errors='coerce')
current_year = datetime.now().year
players['age'] = players['birthDate'].apply(lambda x: current_year - x.year if pd.notnull(x) else None)
players.drop('birthDate', axis=1, inplace=True)
print(players.head())


        bioID  pos  height  weight            college            collegeOther  \
0  abrahta01w    C    74.0     190  George Washington                     NaN   
1  abrossv01w    F    74.0     169        Connecticut                     NaN   
2  adairje01w    C    76.0     197  George Washington                     NaN   
3  adamsda01w  F-C    73.0     239          Texas A&M  Jefferson College (JC)   
4  adamsjo01w    C    75.0     180         New Mexico                     NaN   

   isAlive   age  
0        1  49.0  
1        1  44.0  
2        1  38.0  
3        1  35.0  
4        1  43.0  


# Player Awards Cleaning Up, Award Value Attribution and Player_Teams and Coaches merging


In [53]:
print(player_awards.head(1), '\n')

print(player_awards['award'].unique())


     playerID                               award  year
0  thompti01w  All-Star Game Most Valuable Player     1 

['All-Star Game Most Valuable Player' 'Coach of the Year'
 'Defensive Player of the Year' 'Kim Perrot Sportsmanship'
 'Kim Perrot Sportsmanship Award' 'Most Improved Player'
 'Most Valuable Player' 'Rookie of the Year' 'Sixth Woman of the Year'
 'WNBA Finals Most Valuable Player' 'WNBA All-Decade Team'
 'WNBA All Decade Team Honorable Mention']


## Merge awards_players and players_teams

In [58]:
# We want to establish value to these awards excluding the not so important for winning like Kim Perrot Sportsmanship for example
# Ranking of Awards should be -- MVP, WNBA Finals MVP, Coach of the Year, DPOY, All Star MVP, WNBA All-decade, MIP, 
#WNBA All Decade Team Honorable Mention, ROY, 6WOY, Sportsmanship awards (these 2 should be irrelevant for winning) 10 point system

# Create a award_value table

player_awards['award'] = player_awards['award'].replace(
    'Kim Perrot Sportsmanship', 'Kim Perrot Sportsmanship Award'
)

# Create a award_value table
award_values = {
    "Most Valuable Player": 10,
    "WNBA Finals Most Valuable Player": 9,
    "Coach of the Year": 1,
    "Defensive Player of the Year": 7,
    "All-Star Game Most Valuable Player": 6,
    "WNBA All-Decade Team": 5,
    "Most Improved Player": 4,
    "WNBA All Decade Team Honorable Mention": 3,
    "Rookie of the Year": 2,
    "Sixth Woman of the Year": 1,
    "Kim Perrot Sportsmanship Award": 0,  # irrelevant for winning
}


# Group by award
awards_pivot = player_awards.pivot_table(
    index=['playerID', 'year'],
    columns='award',
    aggfunc='size',
    fill_value=0
).reset_index()

awards_pivot['award_score'] = sum(
    awards_pivot.get(award, 0) * value for award, value in award_values.items()
)

player_teams = pd.merge(player_teams, awards_pivot[['playerID', 'year', 'award_score']], on=['playerID', 'year'], how='left')
player_teams['award_score'].fillna(0, inplace=True)

player_teams.to_csv('merged_players_awards_with_values.csv', index=False)
print(player_teams.head())

     playerID  year  stint tmID  GP  GS  minutes  points  oRebounds  \
0  abrossv01w     2      0  MIN  26  23      846     343         43   
1  abrossv01w     3      0  MIN  27  27      805     314         45   
2  abrossv01w     4      0  MIN  30  25      792     318         44   
3  abrossv01w     5      0  MIN  22  11      462     146         17   
4  abrossv01w     6      0  MIN  31  31      777     304         29   

   dRebounds  ...  PostTurnovers  PostPF  PostfgAttempted  PostfgMade  \
0        131  ...              0       0                0           0   
1        101  ...              0       0                0           0   
2         97  ...              8       8               22           6   
3         57  ...              3       7               23           8   
4         78  ...              0       0                0           0   

   PostftAttempted  PostftMade  PostthreeAttempted  PostthreeMade  PostDQ  \
0                0           0                   0       

## Merge award_players and coaches

In [61]:

awards_pivot_renamed = awards_pivot.rename(columns={'playerID': 'coachID'})
coaches = pd.merge(coaches, awards_pivot_renamed[['coachID', 'year', 'award_score']], on=['coachID', 'year'], how='left')
coaches['award_score'].fillna(0, inplace=True)

coaches.to_csv('merged_coaches_awards.csv', index=False)
print(coaches.head())

      coachID  year tmID  stint  won  lost  post_wins  post_losses  \
0  adamsmi01w     5  WAS      0   17    17          1            2   
1  adubari99w     1  NYL      0   20    12          4            3   
2  adubari99w     2  NYL      0   21    11          3            3   
3  adubari99w     3  NYL      0   18    14          4            4   
4  adubari99w     4  NYL      0   16    18          0            0   

   award_score  
0          0.0  
1          0.0  
2          0.0  
3          0.0  
4          0.0  


" coach_of_year = player_awards[player_awards['award'] == 'Coach of the Year'][['playerID']]\ncoach_of_year.rename(columns={'playerID': 'coachID'}, inplace=True)\ncoaches_updated = pd.merge(coaches, coach_of_year, on=['coachID'], how='left', indicator=True)\ncoaches_updated['coach_of_year'] = coaches_updated['_merge'].apply(lambda x: 1 if x == 'both' else 0)\ncoaches_updated.drop(columns=['_merge'], inplace=True)\n\n# Display the updated coaches DataFrame\nprint(coaches_updated.head(5))\n "